In [1]:
#from google.colab import drive
#drive.mount('/content/drive')

!pip install lpips scikit-image pandas tqdm pillow torch torchvision

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import os
import numpy as np
from PIL import Image
from tqdm import tqdm
from skimage.metrics import peak_signal_noise_ratio as compare_psnr
from skimage.metrics import structural_similarity as compare_ssim
import torch
import lpips
from torchvision import transforms
import pandas as pd

In [3]:
def center_crop(img, target_size=(1000, 640)):
    """Center crop image to target_size."""
    w, h = img.size
    tw, th = target_size
    if w < tw or h < th:
        raise ValueError(f"Image size ({w}x{h}) is smaller than target size ({tw}x{th})")
    left = (w - tw) // 2
    top = (h - th) // 2
    return img.crop((left, top, left + tw, top + th))

def load_image(path, for_lpips=False, apply_crop=True, target_size=(1000, 640)):
    """Normalize image to [0,1] or convert to tensor for LPIPS."""
    img = Image.open(path).convert('RGB')
    if apply_crop: img = center_crop(img, target_size)
    if for_lpips:
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3)
        ])
        return transform(img).unsqueeze(0)
    else:
        return np.array(img).astype(np.float32) / 255.0

In [4]:
def compute_all_metrics(gt_dir, pred_dir, lpips_net='alex', apply_crop=True, target_size=(1000, 640)):
    """
    GT와 PRED 폴더를 직접 비교하여 PSNR, SSIM, LPIPS를 계산합니다.
    """
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    loss_fn = lpips.LPIPS(net=lpips_net).to(device).eval()

    gt_images = sorted([f for f in os.listdir(gt_dir) if f.endswith(('png', 'jpg', 'jpeg')) and not f.startswith('._')])
    pred_images = sorted([f for f in os.listdir(pred_dir) if f.endswith(('png', 'jpg', 'jpeg')) and not f.startswith('._')])

    if not gt_images or not pred_images:
        print(f"    [Warn] No images found! GT: {len(gt_images)}, Pred: {len(pred_images)}")
        return None

    if len(gt_images) != len(pred_images):
        print(f"    [Error] Count mismatch: GT({len(gt_images)}) vs Pred({len(pred_images)})")
        return None

    psnr_list, ssim_list, lpips_list = [], [], []
    pbar = tqdm(zip(gt_images, pred_images), total=len(gt_images), desc='    Processing', leave=False)
    
    for gt_img, pred_img in pbar:
        gt_path = os.path.join(gt_dir, gt_img)
        pred_path = os.path.join(pred_dir, pred_img)
        try:
            gt_np = load_image(gt_path, apply_crop=apply_crop, target_size=target_size)
            pred_np = load_image(pred_path, apply_crop=apply_crop, target_size=target_size)
            
            psnr_val = compare_psnr(gt_np, pred_np, data_range=1.0)
            ssim_val = compare_ssim(gt_np, pred_np, data_range=1.0, channel_axis=2)
            
            gt_tensor = load_image(gt_path, for_lpips=True, apply_crop=apply_crop, target_size=target_size).to(device)
            pred_tensor = load_image(pred_path, for_lpips=True, apply_crop=apply_crop, target_size=target_size).to(device)
            with torch.no_grad():
                lpips_val = loss_fn(gt_tensor, pred_tensor).item()
                
            psnr_list.append(psnr_val)
            ssim_list.append(ssim_val)
            lpips_list.append(lpips_val)
        except Exception as e:
            continue

    if not psnr_list: return None
    return np.mean(psnr_list), np.mean(ssim_list), np.mean(lpips_list)

In [5]:
DIR_BASE = '/Volumes/YHLee/3_Research/26_Capstone2'
SCENES = ["grass", "hydrant", "pillar", "road", "sky", "stair"]
METHODS = ["LongSplat", "3DGS"]
CONDITIONS = ["clean", "snow", "de_snow"]

In [16]:
# === [Option 1] Auto-Combination (Method/Scene/Condition) ===

DIR_BASE = '/Volumes/YHLee/3_Research/26_Capstone2'
SCENES = ["hydrant"]
METHODS = ["3DGS"]
CONDITIONS = ["snow", "de_snow"]

APPLY_CROP = False # Crop 활성화 여부

results = []
for s in SCENES:
    for m in METHODS:
        for c in CONDITIONS:
            print(f"\n>>> Evaluating: {m} | {s} | {c}")
            
            gt_dir = os.path.join(DIR_BASE, m, f"{s}_clean", "renders")
            pred_dir = os.path.join(DIR_BASE, m, f"{s}_{c}", "renders")
            
            if not os.path.exists(gt_dir) or not os.path.exists(pred_dir):
                print(f"    [Skip] Path not found.")
                continue
                
            metrics = compute_all_metrics(gt_dir, pred_dir, apply_crop=APPLY_CROP)
            if metrics:
                psnr, ssim, lpips_val = metrics
                results.append({
                    "Scene": s,
                    "Method": m,  
                    "Condition": c, 
                    "PSNR": psnr, 
                    "SSIM": ssim, 
                    "LPIPS": lpips_val
                })
                print(f"    => PSNR: {psnr:.3f} | SSIM: {ssim:.3f} | LPIPS: {lpips_val:.3f}")

if results:
    df = pd.DataFrame(results)
    csv_path = os.path.join(DIR_BASE, 'metrics_results_v0_auto.csv')
    df.to_csv(csv_path, index=False)
    print("\n" + "="*50 + "\n            AUTO-COMBINATION SUMMARY\n" + "="*50)
    print(f" >> Saved to: {csv_path}")
    display(df)


>>> Evaluating: 3DGS | hydrant | snow
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /Users/yhlee/Library/Python/3.9/lib/python/site-packages/lpips/weights/v0.1/alex.pth


/Users/yhlee/Library/Python/3.9/lib/python/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/yhlee/Library/Python/3.9/lib/python/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


    [Error] Count mismatch: GT(2) vs Pred(18)

>>> Evaluating: 3DGS | hydrant | de_snow
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /Users/yhlee/Library/Python/3.9/lib/python/site-packages/lpips/weights/v0.1/alex.pth
